# SpecMine results for the paper

This notebook fits fresh Logistic Regression and Random Forest baselines with confidence sample weights, then evaluates them on a seeded, repository-disjoint holdout. It uses the test labels in `examples/fake_labels.csv`; its placeholder confidence values are test data, not verified assessments. No pre-trained model files are required. All reported metrics describe agreement with the supplied labels. The small test sample means estimates and feature rankings may have substantial variance.

Figures are saved as 300-dpi PNG files in `notebooks/figures/`. Metadata and feature tables are loaded cache-first from `notebooks/data`; a missing parquet file is downloaded from Hugging Face and saved there.

In [ ]:
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

from specmine.data import load_data, load_feature_data
from specmine.evaluate import evaluate_model
from specmine.labels import SPEC_ID_COLUMN, read_labels_csv
from specmine.train import (
    ID_COLUMN,
    REPOSITORY_COLUMN,
    prepare_training_data,
    repository_disjoint_split,
    train_models,
)

SEED = 42
TEST_SIZE = 0.2
BOOTSTRAP_REPLICATES = 2000
LABELS_CSV = project_root / "examples" / "fake_labels.csv"
FIGURE_DIR = project_root / "notebooks" / "figures"
DATA_DIR = project_root / "notebooks" / "data"
MODEL_NAMES = ("logistic_regression", "random_forest")
MODEL_COLORS = {"logistic_regression": "#2878B5", "random_forest": "#D9782D"}
LABEL_COLORS = {0: "#3B6FB6", 1: "#D9782D"}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.dpi": 120,
        "savefig.dpi": 300,
    }
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Labels: {LABELS_CSV.relative_to(project_root)}")
print(f"Figures: {FIGURE_DIR.relative_to(project_root)}")

## 1. Dataset composition

These plots show how many matched specs belong to each supplied class and how those labels are distributed across source repositories. The table also includes ambiguous (blank) labels so that exclusions remain visible.

In [ ]:
def load_metadata():
    return load_data(seed=SEED, data_dir=DATA_DIR)


def load_labeled_features(spec_ids):
    return load_feature_data(
        pd.DataFrame({ID_COLUMN: spec_ids}),
        data_dir=DATA_DIR,
    )


labels_frame = read_labels_csv(LABELS_CSV)
metadata = load_metadata()
metadata_lookup = metadata[[ID_COLUMN, REPOSITORY_COLUMN]].drop_duplicates()
matched_labels = labels_frame.merge(
    metadata_lookup,
    how="left",
    left_on=SPEC_ID_COLUMN,
    right_on=ID_COLUMN,
    validate="one_to_one",
)
unmatched_label_count = int(matched_labels[REPOSITORY_COLUMN].isna().sum())
if unmatched_label_count:
    print(f"Labels not matched to metadata: {unmatched_label_count}")
composition = matched_labels.loc[matched_labels[REPOSITORY_COLUMN].notna()].copy()
composition["class_name"] = (
    composition["label"].map({0: "Label 0", 1: "Label 1"}).fillna("Ambiguous")
)
repository_class_counts = (
    composition.groupby([REPOSITORY_COLUMN, "class_name"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Label 0", "Label 1", "Ambiguous"], fill_value=0)
    .sort_index()
)
display(repository_class_counts.rename_axis("source_repo").reset_index())
display(
    composition["class_name"]
    .value_counts()
    .reindex(["Label 0", "Label 1", "Ambiguous"], fill_value=0)
    .rename("n")
    .rename_axis("class")
    .to_frame()
)

binary_composition = composition.loc[composition["label"].notna()].copy()
binary_composition["label"] = binary_composition["label"].astype(int)

The first figure makes the overall class balance explicit; its bar labels are the exact matched-spec counts, not percentages.

In [ ]:
def add_sample_note(fig, frames):
    note_parts = []
    for name, frame in frames.items():
        class_counts = frame["label"].value_counts().reindex([0, 1], fill_value=0)
        repo_counts = (
            frame.groupby([REPOSITORY_COLUMN, "label"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=[0, 1], fill_value=0)
            .sort_index()
        )
        repo_text = ", ".join(
            f"{repo} (0={row[0]}, 1={row[1]})" for repo, row in repo_counts.iterrows()
        )
        note_parts.append(
            f"{name}: n={len(frame)}; label 0={class_counts.loc[0]}, "
            f"label 1={class_counts.loc[1]}; per-repo label counts: {repo_text}"
        )
    note = textwrap.fill(" | ".join(note_parts), width=170)
    lines = note.count("\n") + 1
    fig.subplots_adjust(bottom=min(0.48, 0.12 + 0.025 * lines))
    fig.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=6.5)


class_counts = binary_composition["label"].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
bars = ax.bar(
    ["Label 0", "Label 1"],
    [class_counts.loc[0], class_counts.loc[1]],
    color=[LABEL_COLORS[0], LABEL_COLORS[1]],
    width=0.62,
)
ax.bar_label(
    bars, labels=[f"n={class_counts.loc[0]}", f"n={class_counts.loc[1]}"], padding=3
)
ax.set_ylabel("Number of labeled specs")
ax.set_title("Manually labeled dataset composition")
ax.set_ylim(0, max(class_counts.max() * 1.15, 1))
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"matched labels": binary_composition})
fig.savefig(FIGURE_DIR / "dataset_label_counts.png", dpi=300, bbox_inches="tight")
plt.show()

The repository chart reveals whether either class is concentrated in a small set of projects. Counts are printed on each nonzero stack segment.

In [ ]:
plot_repo_counts = (
    pd.crosstab(binary_composition[REPOSITORY_COLUMN], binary_composition["label"])
    .reindex(columns=[0, 1], fill_value=0)
    .sort_index()
)
fig, ax = plt.subplots(figsize=(max(9, 0.48 * len(plot_repo_counts)), 6))
bottom = np.zeros(len(plot_repo_counts))
for label in [0, 1]:
    values = plot_repo_counts[label].to_numpy()
    bars = ax.bar(
        plot_repo_counts.index,
        values,
        bottom=bottom,
        color=LABEL_COLORS[label],
        label=f"Label {label}",
    )
    for bar, value, base in zip(bars, values, bottom):
        if value:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                base + value / 2,
                str(value),
                ha="center",
                va="center",
                fontsize=7,
                color="white",
            )
    bottom += values
ax.set_ylabel("Number of labeled specs")
ax.set_xlabel("Source repository")
ax.set_title("Label counts by source repository")
ax.tick_params(axis="x", labelrotation=70)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"matched labels": binary_composition})
fig.savefig(
    FIGURE_DIR / "dataset_labels_by_repository.png", dpi=300, bbox_inches="tight"
)
plt.show()

## 2. Split sanity check

The split below calls the same `repository_disjoint_split` helper used by training, with seed 42 and test size 0.2. The printed repository lists and zero-overlap assertion make leakage visible; the class counts show any imbalance caused by holding out whole repositories.

In [ ]:
labeled_ids = labels_frame.loc[labels_frame["label"].notna(), SPEC_ID_COLUMN]
raw_labeled_features = load_labeled_features(labeled_ids)
training_features, training_labels, training_repositories, training_confidences = (
    prepare_training_data(
        metadata,
        labels_frame,
        feature_data=raw_labeled_features,
        seed=SEED,
    )
)
train_indices, test_indices = repository_disjoint_split(
    training_features,
    training_labels,
    training_repositories,
    test_size=TEST_SIZE,
    seed=SEED,
)
training_results = train_models(
    training_features,
    training_labels,
    training_repositories,
    confidence_scores=training_confidences,
    model_names=MODEL_NAMES,
    test_size=TEST_SIZE,
    seed=SEED,
)
models = {name: result["model"] for name, result in training_results.items()}

train_features = training_features.iloc[train_indices].reset_index(drop=True)
test_features = training_features.iloc[test_indices].reset_index(drop=True)
train_frame = train_features.copy()
train_frame["label"] = training_labels.iloc[train_indices].to_numpy()
train_frame[REPOSITORY_COLUMN] = training_repositories.iloc[train_indices].to_numpy()
test_frame = test_features.copy()
test_frame["label"] = training_labels.iloc[test_indices].to_numpy()
test_frame[REPOSITORY_COLUMN] = training_repositories.iloc[test_indices].to_numpy()

train_repos = sorted(train_frame[REPOSITORY_COLUMN].unique(), key=str)
test_repos = sorted(test_frame[REPOSITORY_COLUMN].unique(), key=str)
shared_repos = set(train_repos) & set(test_repos)
assert not shared_repos, f"Repository leakage detected: {sorted(shared_repos)}"
print("Train repositories:", train_repos)
print("Test repositories:", test_repos)
print("Shared repositories:", sorted(shared_repos))
split_balance = pd.DataFrame(
    {
        "train": train_frame["label"].value_counts().reindex([0, 1], fill_value=0),
        "test": test_frame["label"].value_counts().reindex([0, 1], fill_value=0),
    }
).rename_axis("label")
display(split_balance)
print(f"Train: {len(train_frame)} specs / {len(train_repos)} repos")
print(f"Test:  {len(test_frame)} specs / {len(test_repos)} repos")

This grouped bar chart shows how repository holdout changes class counts. It reports counts rather than normalized proportions so the small test support is not obscured.

In [ ]:
positions = np.arange(2)
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4.5))
for offset, split_name, color, frame in [
    (-width / 2, "Train", "#527A9E", train_frame),
    (width / 2, "Test", "#D9782D", test_frame),
]:
    values = frame["label"].value_counts().reindex([0, 1], fill_value=0).to_numpy()
    bars = ax.bar(positions + offset, values, width, label=split_name, color=color)
    ax.bar_label(bars, labels=[f"n={value}" for value in values], padding=2, fontsize=8)
ax.set_xticks(positions, ["Label 0", "Label 1"])
ax.set_ylabel("Number of specs")
ax.set_title("Class balance in the repository-disjoint split")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"train": train_frame, "test": test_frame})
fig.savefig(FIGURE_DIR / "split_class_balance.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Model comparison

The baselines are fit on the training repositories with confidence sample weights and scored on exactly the test rows selected above. Error bars are percentile 95% intervals from a repository-cluster bootstrap, resampling whole repositories to account for within-repository dependence. With few repositories they remain descriptive and may be imprecise. AUC is omitted when a resample contains only one class. The table includes raw confusion counts beside each metric.

In [ ]:
evaluation_results = {}
prediction_frames = {}
for model_name, model in models.items():
    evaluated = evaluate_model(model, test_frame)
    if evaluated["metrics"] is None:
        raise RuntimeError(f"Evaluation did not return metrics for {model_name}.")
    evaluation_results[model_name] = evaluated
    prediction_frames[model_name] = pd.DataFrame(evaluated["predictions"])


def metric_values(y_true, y_pred, y_prob):
    values = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
    }
    if np.unique(y_true).size == 2:
        values["roc_auc"] = roc_auc_score(y_true, y_prob)
    return values


def bootstrap_intervals(y_true, y_pred, y_prob, groups, *, replicates, seed):
    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    if len(groups) != len(y_true) or len(y_true) == 0:
        raise ValueError("groups and labels must have the same nonzero length")
    group_rows = [np.flatnonzero(groups == group) for group in pd.unique(groups)]
    draws = {name: [] for name in ["accuracy", "precision", "recall", "f1", "roc_auc"]}
    for _ in range(replicates):
        sampled_groups = rng.integers(0, len(group_rows), size=len(group_rows))
        indices = np.concatenate([group_rows[index] for index in sampled_groups])
        sample_metrics = metric_values(
            y_true[indices], y_pred[indices], y_prob[indices]
        )
        for name, value in sample_metrics.items():
            if np.isfinite(value):
                draws[name].append(value)
    return {
        name: tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)
        for name, values in draws.items()
    }


summary_rows = []
bootstrap_results = {}
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
for model_index, model_name in enumerate(MODEL_NAMES):
    prediction = prediction_frames[model_name]
    y_true = prediction["y_true"].to_numpy(dtype=int)
    y_pred = prediction["y_pred"].to_numpy(dtype=int)
    y_prob = prediction["y_prob"].to_numpy(dtype=float)
    metrics = evaluation_results[model_name]["metrics"]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    intervals = bootstrap_intervals(
        y_true,
        y_pred,
        y_prob,
        test_frame[REPOSITORY_COLUMN].to_numpy(),
        replicates=BOOTSTRAP_REPLICATES,
        seed=SEED + model_index,
    )
    bootstrap_results[model_name] = intervals
    row = {
        "model": model_name,
        "n_test": len(prediction),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
    }
    row.update({name: metrics[name] for name in metric_names})
    row.update({f"{name}_95%_CI": intervals[name] for name in metric_names})
    summary_rows.append(row)
summary_table = pd.DataFrame(summary_rows).set_index("model")
display(summary_table)
print(
    f"Bootstrap replicates: {BOOTSTRAP_REPLICATES}; test labels: {test_frame['label'].value_counts().sort_index().to_dict()}"
)

The grouped bars compare the five requested metrics on a common 0–1 scale. Error bars show bootstrap percentile intervals; a missing ROC-AUC interval means the held-out sample did not contain both classes.

In [ ]:
x = np.arange(len(metric_names))
width = 0.36
fig, ax = plt.subplots(figsize=(9, 5.5))
for offset, model_name in zip([-width / 2, width / 2], MODEL_NAMES):
    point_values = np.array(
        [evaluation_results[model_name]["metrics"][metric] for metric in metric_names],
        dtype=float,
    )
    lower = []
    upper = []
    for metric, point in zip(metric_names, point_values):
        low, high = bootstrap_results[model_name][metric]
        if np.isfinite(point) and np.isfinite(low) and np.isfinite(high):
            lower.append(max(point - low, 0))
            upper.append(max(high - point, 0))
        else:
            lower.append(0)
            upper.append(0)
    bars = ax.bar(
        x + offset,
        point_values,
        width,
        label=model_name.replace("_", " ").title(),
        color=MODEL_COLORS[model_name],
        yerr=np.array([lower, upper]),
        capsize=3,
    )
    for bar, value in zip(bars, point_values):
        if np.isfinite(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + 0.025,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=7,
            )
ax.set_xticks(x, [name.replace("_", " ").title() for name in metric_names])
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Baseline performance on the repository-disjoint test split")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"test": test_frame})
fig.savefig(FIGURE_DIR / "model_metric_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Confusion matrices

The heatmaps show raw test-set counts (true labels on rows, predictions on columns), preserving the practical scale of each error type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, model_name in zip(axes, MODEL_NAMES):
    matrix = np.asarray(evaluation_results[model_name]["metrics"]["confusion_matrix"])
    image = ax.imshow(matrix, cmap="Blues", vmin=0)
    for row in range(2):
        for column in range(2):
            ax.text(
                column,
                row,
                str(matrix[row, column]),
                ha="center",
                va="center",
                fontsize=12,
                color="black",
            )
    ax.set_xticks([0, 1], ["Pred 0", "Pred 1"])
    ax.set_yticks([0, 1], ["True 0", "True 1"])
    ax.set_title(model_name.replace("_", " ").title())
fig.suptitle("Confusion matrices (raw counts)")
add_sample_note(fig, {"test": test_frame})
fig.savefig(FIGURE_DIR / "confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. ROC and precision–recall curves

The ROC plot uses the diagonal no-skill reference; the precision–recall plot uses test prevalence as its no-skill reference. PR performance is especially informative when the test labels are imbalanced. Curves are skipped with an explicit note if the held-out split contains only one class.

In [ ]:
y_test = test_frame["label"].to_numpy(dtype=int)
if np.unique(y_test).size == 2:
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for model_name in MODEL_NAMES:
        prediction = prediction_frames[model_name]
        fpr, tpr, _ = roc_curve(y_test, prediction["y_prob"].to_numpy())
        auc = evaluation_results[model_name]["metrics"]["roc_auc"]
        ax.plot(
            fpr,
            tpr,
            lw=2,
            color=MODEL_COLORS[model_name],
            label=f"{model_name.replace('_', ' ').title()} (AUC={auc:.2f})",
        )
    ax.plot([0, 1], [0, 1], linestyle="--", color="#666666", label="No-skill")
    ax.set(
        xlabel="False positive rate",
        ylabel="True positive rate",
        title="ROC curves on the held-out test split",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.legend(frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"test": test_frame})
    fig.savefig(FIGURE_DIR / "roc_curves.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("ROC curves are undefined: the repository-disjoint test split has one class.")

In [ ]:
if np.unique(y_test).size == 2:
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for model_name in MODEL_NAMES:
        prediction = prediction_frames[model_name]
        precision, recall, _ = precision_recall_curve(
            y_test, prediction["y_prob"].to_numpy()
        )
        average_precision = average_precision_score(
            y_test, prediction["y_prob"].to_numpy()
        )
        ax.plot(
            recall,
            precision,
            lw=2,
            color=MODEL_COLORS[model_name],
            label=f"{model_name.replace('_', ' ').title()} (AP={average_precision:.2f})",
        )
    prevalence = float(np.mean(y_test))
    ax.axhline(
        prevalence,
        linestyle="--",
        color="#666666",
        label=f"No-skill (prevalence={prevalence:.2f})",
    )
    ax.set(
        xlabel="Recall",
        ylabel="Precision",
        title="Precision–recall curves on the held-out test split",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.legend(frameon=False, loc="best")
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"test": test_frame})
    fig.savefig(
        FIGURE_DIR / "precision_recall_curves.png", dpi=300, bbox_inches="tight"
    )
    plt.show()
else:
    print(
        "Precision–recall curves are undefined: the repository-disjoint test split has one class."
    )

## 6. Calibration and score distribution

The Logistic Regression probability histogram separates test specs by true label. The calibration plot is drawn only when the test split has at least 20 examples and at least five examples from each class; otherwise the notebook reports that support is too small.

In [ ]:
lr_predictions = prediction_frames["logistic_regression"]
fig, ax = plt.subplots(figsize=(7, 5))
for label in [0, 1]:
    probabilities = lr_predictions.loc[lr_predictions["y_true"].eq(label), "y_prob"]
    ax.hist(
        probabilities,
        bins=np.linspace(0, 1, 11),
        alpha=0.62,
        color=LABEL_COLORS[label],
        edgecolor="white",
        label=f"True label {label} (n={len(probabilities)})",
    )
ax.set(
    xlabel="Predicted probability of label 1",
    ylabel="Number of specs",
    title="Logistic Regression test probability distribution",
    xlim=(0, 1),
)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"test": test_frame})
fig.savefig(
    FIGURE_DIR / "logistic_probability_distribution.png", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
test_class_counts = test_frame["label"].value_counts()
if (
    len(test_frame) >= 20
    and min(test_class_counts.get(0, 0), test_class_counts.get(1, 0)) >= 5
):
    n_bins = min(5, max(2, len(test_frame) // 5))
    observed, predicted = calibration_curve(
        lr_predictions["y_true"],
        lr_predictions["y_prob"],
        n_bins=n_bins,
        strategy="quantile",
    )
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(
        [0, 1], [0, 1], linestyle="--", color="#666666", label="Perfect calibration"
    )
    ax.plot(
        predicted,
        observed,
        marker="o",
        color=MODEL_COLORS["logistic_regression"],
        label="Logistic Regression",
    )
    ax.set(
        xlabel="Mean predicted probability",
        ylabel="Observed fraction of label 1",
        title=f"Logistic Regression calibration ({n_bins} quantile bins)",
        xlim=(0, 1),
        ylim=(0, 1),
    )
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"test": test_frame})
    fig.savefig(
        FIGURE_DIR / "logistic_calibration_curve.png", dpi=300, bbox_inches="tight"
    )
    plt.show()
else:
    print(
        "Calibration curve omitted: require at least 20 test specs and at least 5 from each class."
    )

## 7. Feature importance

These are the top ten Random Forest impurity importances and Logistic Regression signed coefficients from the saved training estimators. With roughly 100 labels and correlated engineered features, rankings and coefficient magnitudes can have substantial small-sample variance; treat them as descriptive, not causal.

In [ ]:
rf_pipeline = models["random_forest"]
rf_classifier = rf_pipeline.named_steps["classifier"]
rf_feature_names = np.asarray(rf_pipeline.feature_names_in_, dtype=str)
rf_importance = pd.Series(rf_classifier.feature_importances_, index=rf_feature_names)
rf_top = rf_importance.nlargest(10).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(rf_top.index, rf_top.values, color=MODEL_COLORS["random_forest"])
ax.set(xlabel="Mean impurity importance", title="Top 10 Random Forest features")
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"train": train_frame})
fig.savefig(
    FIGURE_DIR / "random_forest_top10_importances.png", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
lr_pipeline = models["logistic_regression"]
lr_classifier = lr_pipeline.named_steps["classifier"]
lr_feature_names = np.asarray(lr_pipeline.feature_names_in_, dtype=str)
lr_coefficients = pd.Series(lr_classifier.coef_[0], index=lr_feature_names)
lr_top = lr_coefficients.loc[lr_coefficients.abs().nlargest(10).index].sort_values()
coefficient_colors = ["#3B6FB6" if value < 0 else "#D9782D" for value in lr_top.values]
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(lr_top.index, lr_top.values, color=coefficient_colors)
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set(
    xlabel="Logistic Regression coefficient (scaled features)",
    title="Top 10 Logistic Regression coefficients by absolute value",
)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"train": train_frame})
fig.savefig(
    FIGURE_DIR / "logistic_regression_top10_coefficients.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 8. Error analysis

The table lists every held-out misclassification with its true label, predicted probability, and source repository. Per-repository error counts and rates indicate whether errors appear concentrated; the flag requires at least two errors and assigns concentration when one repo contains at least half, a descriptive heuristic rather than a significance test.

In [ ]:
prediction_rows = []
error_rows = []
error_by_model = {}
for model_name in MODEL_NAMES:
    predictions = prediction_frames[model_name].copy()
    predictions["source_repo"] = test_frame[REPOSITORY_COLUMN].to_numpy()
    predictions["predicted_confidence"] = (
        np.maximum(predictions["y_prob"], 1 - predictions["y_prob"]) * 100
    )
    predictions["model"] = model_name
    predictions = predictions.rename(
        columns={
            "id": "spec_id",
            "y_true": "true_label",
            "y_prob": "predicted_probability",
            "y_pred": "predicted_label",
        }
    )
    columns = [
        "model",
        "spec_id",
        "true_label",
        "predicted_probability",
        "predicted_label",
        "predicted_confidence",
        "source_repo",
    ]
    prediction_rows.append(predictions[columns])
    errors = predictions.loc[
        predictions["true_label"].ne(predictions["predicted_label"])
    ].copy()
    error_rows.append(errors[columns])
    error_by_model[model_name] = errors

prediction_table = pd.concat(prediction_rows, ignore_index=True)
display(
    prediction_table.sort_values(["model", "source_repo", "spec_id"]).style.format(
        {
            "predicted_probability": "{:.3f}",
            "predicted_confidence": "{:.1f}%",
        }
    )
)

error_table = pd.concat(error_rows, ignore_index=True)
if error_table.empty:
    print("No misclassified test specs for either model.")
else:
    display(
        error_table.sort_values(["model", "source_repo", "spec_id"]).style.format(
            {
                "predicted_probability": "{:.3f}",
                "predicted_confidence": "{:.1f}%",
            }
        )
    )

for model_name, errors in error_by_model.items():
    repo_errors = errors["source_repo"].value_counts().sort_values(ascending=False)
    total_errors = len(errors)
    if total_errors == 0:
        print(f"{model_name}: no errors; repository clustering is not applicable.")
        continue
    error_counts = (
        test_frame.groupby(REPOSITORY_COLUMN).size().rename("n_test").to_frame()
    )
    error_counts["n_errors"] = errors["source_repo"].value_counts()
    error_counts["n_errors"] = error_counts["n_errors"].fillna(0).astype(int)
    error_counts["error_rate"] = error_counts["n_errors"] / error_counts["n_test"]
    error_counts = error_counts.sort_values(["n_errors", "error_rate"], ascending=False)
    display(error_counts.rename_axis("source_repo").reset_index())
    leading_repo = repo_errors.index[0]
    leading_count = int(repo_errors.iloc[0])
    clustered = (
        total_errors >= 2 and leading_count >= 2 and leading_count / total_errors >= 0.5
    )
    print(
        f"{model_name}: {'FLAG — errors concentrated' if clustered else 'No majority-repo concentration'}; "
        f"{leading_repo} has {leading_count}/{total_errors} errors."
    )

print("Saved figures:")
for figure_path in sorted(FIGURE_DIR.glob("*.png")):
    print(figure_path.relative_to(project_root))